In [1]:
from pathlib import Path

import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 300)
pd.set_option("display.max_colwidth", 300)

PROJECT_DIR = Path("..").resolve()

REPORTS_DIR = (
    PROJECT_DIR
    / "reports"
    / "tables"
)

INTERIM_DIR = (
    PROJECT_DIR
    / "data"
    / "public_v2"
    / "interim"
)

In [2]:
dictionary = pd.read_csv(
    REPORTS_DIR
    / "feature_dictionary_validation.csv"
)

rico_features = pd.read_csv(
    INTERIM_DIR
    / "rico_structural_features.csv"
)

uicrit_rico_proxy = pd.read_csv(
    INTERIM_DIR
    / "uicrit_rico_proxy_dataset.csv"
)

catching_features = pd.read_csv(
    INTERIM_DIR
    / "catching_static_site_features.csv"
)

catching_validation = pd.read_csv(
    INTERIM_DIR
    / "catching_static_validation_dataset.csv"
)

In [3]:
print(
    "Dictionary:",
    dictionary.shape
)

print(
    "RICO X:",
    rico_features.shape
)

print(
    "UICrit/RICO X+Y:",
    uicrit_rico_proxy.shape
)

print(
    "Catching X:",
    catching_features.shape
)

print(
    "Catching X+Y:",
    catching_validation.shape
)

Dictionary: (48, 10)
RICO X: (1000, 14)
UICrit/RICO X+Y: (1000, 23)
Catching X: (2, 12)
Catching X+Y: (2, 23)


In [4]:
canonical_x = (
    dictionary[
        dictionary[
            "rol_ml"
        ]
        ==
        "X"
    ]
    .copy()
)

print(
    "X canónicas:",
    len(canonical_x)
)

assert len(canonical_x) == 40

X canónicas: 40


In [5]:
display(
    canonical_x[
        [
            col
            for col in [
                "variable",
                "categoria",
                "rol_ml",
                "captura_extension",
                "estado_final"
            ]
            if col in canonical_x.columns
        ]
    ]
)

,variable,categoria,rol_ml,captura_extension,estado_final
4,task_time_s,TEMPORAL,X,DIRECTA,RESPALDADA_POR_DATOS_PUBLICOS
5,mean_response_time_ms,TEMPORAL,X,DERIVADA,RECOLECCION_PROPIA
6,max_response_time_ms,TEMPORAL,X,DERIVADA,RECOLECCION_PROPIA
7,mean_pre_action_pause_s,TEMPORAL,X,DERIVADA,RESPALDADA_POR_DATOS_PUBLICOS
8,click_count,INTERACCION,X,DIRECTA,RESPALDADA_POR_DATOS_PUBLICOS
9,keypress_count,INTERACCION,X,DIRECTA,RESPALDADA_POR_DATOS_PUBLICOS
10,scroll_count,INTERACCION,X,DIRECTA,RESPALDADA_POR_DATOS_PUBLICOS
11,backtracking_count,INTERACCION,X,DERIVADA,RESPALDADA_POR_DATOS_PUBLICOS
12,repeated_action_count,INTERACCION,X,DERIVADA,RESPALDADA_POR_DATOS_PUBLICOS
13,navigation_steps,INTERACCION,X,DERIVADA,RESPALDADA_POR_DATOS_PUBLICOS


In [6]:
source_registry = pd.DataFrame([
    {
        "source_dataset":
            "UICrit_RICO",

        "n_instances":
            len(
                uicrit_rico_proxy
            ),

        "feature_modality":
            "ANDROID_UI_HIERARCHY",

        "label_type":
            "RATING_PROXY",

        "direct_nielsen":
            False,

        "primary_training":
            False
    },

    {
        "source_dataset":
            "Catching_UX_Flaws",

        "n_instances":
            len(
                catching_validation
            ),

        "feature_modality":
            "STATIC_HTML_SOURCE",

        "label_type":
            "LLM_DIRECT",

        "direct_nielsen":
            True,

        "primary_training":
            False
    }
])

source_registry

,source_dataset,n_instances,feature_modality,label_type,direct_nielsen,primary_training
0,UICrit_RICO,1000,ANDROID_UI_HIERARCHY,RATING_PROXY,False,False
1,Catching_UX_Flaws,2,STATIC_HTML_SOURCE,LLM_DIRECT,True,False


In [7]:
compatibility_levels = pd.DataFrame([
    {
        "compatibility_level":
            "EXACT",

        "can_be_renamed_canonical":
            True,

        "can_share_feature_space":
            True,

        "definition":
            "Misma operacionalización, unidad y semántica de medición."
    },

    {
        "compatibility_level":
            "PARTIAL",

        "can_be_renamed_canonical":
            False,

        "can_share_feature_space":
            False,

        "definition":
            "Permite derivar información relacionada, pero la operacionalización no es idéntica."
    },

    {
        "compatibility_level":
            "CONSTRUCT_PROXY",

        "can_be_renamed_canonical":
            False,

        "can_share_feature_space":
            False,

        "definition":
            "Mide un constructo relacionado pero mediante otra plataforma o representación."
    },

    {
        "compatibility_level":
            "NO_MAPPING",

        "can_be_renamed_canonical":
            False,

        "can_share_feature_space":
            False,

        "definition":
            "No existe señal equivalente defendible."
    }
])

compatibility_levels

,compatibility_level,can_be_renamed_canonical,can_share_feature_space,definition
0,EXACT,True,True,"Misma operacionalización, unidad y semántica de medición."
1,PARTIAL,False,False,"Permite derivar información relacionada, pero la operacionalización no es idéntica."
2,CONSTRUCT_PROXY,False,False,Mide un constructo relacionado pero mediante otra plataforma o representación.
3,NO_MAPPING,False,False,No existe señal equivalente defendible.


In [8]:
mapping_records = [
    {
        "source_dataset":
            "UICrit_RICO",

        "source_variable":
            "rico_max_tree_depth",

        "canonical_variable":
            "navigation_depth",

        "compatibility_level":
            "CONSTRUCT_PROXY",

        "possible_heuristic":
            "H6",

        "reason":
            "Profundidad del árbol Android no representa pasos de navegación web."
    },

    {
        "source_dataset":
            "UICrit_RICO",

        "source_variable":
            "rico_interactive_label_proxy_ratio",

        "canonical_variable":
            "accessible_name_ratio",

        "compatibility_level":
            "CONSTRUCT_PROXY",

        "possible_heuristic":
            "H6",

        "reason":
            "text/content-desc de Android no equivale al Accessible Name calculado por el navegador."
    },

    {
        "source_dataset":
            "UICrit_RICO",

        "source_variable":
            "rico_interactive_label_proxy_ratio",

        "canonical_variable":
            "visible_labels_ratio",

        "compatibility_level":
            "CONSTRUCT_PROXY",

        "possible_heuristic":
            "H6",

        "reason":
            "La jerarquía Android no reproduce visibilidad y etiquetado de controles HTML."
    },

    {
        "source_dataset":
            "UICrit_RICO",

        "source_variable":
            "rico_unlabeled_interactive_count",

        "canonical_variable":
            "unlabeled_controls_count",

        "compatibility_level":
            "CONSTRUCT_PROXY",

        "possible_heuristic":
            "H6",

        "reason":
            "El concepto está relacionado, pero las reglas de accesibilidad Android y web son distintas."
    },

    {
        "source_dataset":
            "UICrit_RICO",

        "source_variable":
            "rico_class_diversity_ratio",

        "canonical_variable":
            "component_style_consistency_ratio",

        "compatibility_level":
            "CONSTRUCT_PROXY",

        "possible_heuristic":
            "H4",

        "reason":
            "Clases Android no representan estilos/componentes CSS."
    }
]

In [9]:
mapping_records += [
    {
        "source_dataset":
            "Catching_UX_Flaws",

        "source_variable":
            "static_max_dom_depth",

        "canonical_variable":
            "navigation_depth",

        "compatibility_level":
            "CONSTRUCT_PROXY",

        "possible_heuristic":
            "H6",

        "reason":
            "Profundidad del DOM estático no equivale a profundidad de navegación."
    },

    {
        "source_dataset":
            "Catching_UX_Flaws",

        "source_variable":
            "static_control_label_proxy_ratio",

        "canonical_variable":
            "accessible_name_ratio",

        "compatibility_level":
            "CONSTRUCT_PROXY",

        "possible_heuristic":
            "H6",

        "reason":
            "Etiquetas detectadas en HTML fuente no reproducen el algoritmo Accessible Name."
    },

    {
        "source_dataset":
            "Catching_UX_Flaws",

        "source_variable":
            "static_control_label_proxy_ratio",

        "canonical_variable":
            "visible_labels_ratio",

        "compatibility_level":
            "CONSTRUCT_PROXY",

        "possible_heuristic":
            "H6",

        "reason":
            "Sin renderizado CSS no se puede determinar visibilidad real."
    },

    {
        "source_dataset":
            "Catching_UX_Flaws",

        "source_variable":
            "static_native_validation_ratio",

        "canonical_variable":
            "input_validation_present",

        "compatibility_level":
            "PARTIAL",

        "possible_heuristic":
            "H5",

        "reason":
            "required/pattern/min/max permiten detectar validación HTML nativa, pero la variable fuente es un ratio y no cubre toda validación runtime."
    },

    {
        "source_dataset":
            "Catching_UX_Flaws",

        "source_variable":
            "static_mean_class_reuse_ratio",

        "canonical_variable":
            "css_class_consistency_ratio",

        "compatibility_level":
            "CONSTRUCT_PROXY",

        "possible_heuristic":
            "H4",

        "reason":
            "Reutilización de clases no equivale a consistencia CSS."
    },

    {
        "source_dataset":
            "Catching_UX_Flaws",

        "source_variable":
            "static_mean_class_reuse_ratio",

        "canonical_variable":
            "component_style_consistency_ratio",

        "compatibility_level":
            "CONSTRUCT_PROXY",

        "possible_heuristic":
            "H4",

        "reason":
            "La presencia repetida de clases no demuestra consistencia visual renderizada."
    }
]

In [10]:
declared_mappings = pd.DataFrame(
    mapping_records
)

display(
    declared_mappings
)

,source_dataset,source_variable,canonical_variable,compatibility_level,possible_heuristic,reason
0,UICrit_RICO,rico_max_tree_depth,navigation_depth,CONSTRUCT_PROXY,H6,Profundidad del árbol Android no representa pasos de navegación web.
1,UICrit_RICO,rico_interactive_label_proxy_ratio,accessible_name_ratio,CONSTRUCT_PROXY,H6,text/content-desc de Android no equivale al Accessible Name calculado por el navegador.
2,UICrit_RICO,rico_interactive_label_proxy_ratio,visible_labels_ratio,CONSTRUCT_PROXY,H6,La jerarquía Android no reproduce visibilidad y etiquetado de controles HTML.
3,UICrit_RICO,rico_unlabeled_interactive_count,unlabeled_controls_count,CONSTRUCT_PROXY,H6,"El concepto está relacionado, pero las reglas de accesibilidad Android y web son distintas."
4,UICrit_RICO,rico_class_diversity_ratio,component_style_consistency_ratio,CONSTRUCT_PROXY,H4,Clases Android no representan estilos/componentes CSS.
5,Catching_UX_Flaws,static_max_dom_depth,navigation_depth,CONSTRUCT_PROXY,H6,Profundidad del DOM estático no equivale a profundidad de navegación.
6,Catching_UX_Flaws,static_control_label_proxy_ratio,accessible_name_ratio,CONSTRUCT_PROXY,H6,Etiquetas detectadas en HTML fuente no reproducen el algoritmo Accessible Name.
7,Catching_UX_Flaws,static_control_label_proxy_ratio,visible_labels_ratio,CONSTRUCT_PROXY,H6,Sin renderizado CSS no se puede determinar visibilidad real.
8,Catching_UX_Flaws,static_native_validation_ratio,input_validation_present,PARTIAL,H5,"required/pattern/min/max permiten detectar validación HTML nativa, pero la variable fuente es un ratio y no cubre toda validación runtime."
9,Catching_UX_Flaws,static_mean_class_reuse_ratio,css_class_consistency_ratio,CONSTRUCT_PROXY,H4,Reutilización de clases no equivale a consistencia CSS.


In [11]:
canonical_names = set(
    canonical_x[
        "variable"
    ]
)

rico_names = set(
    rico_features.columns
)

catching_names = set(
    catching_features.columns
)

In [12]:
for _, row in declared_mappings.iterrows():

    assert (
        row[
            "canonical_variable"
        ]
        in canonical_names
    ), row[
        "canonical_variable"
    ]

    if (
        row[
            "source_dataset"
        ]
        ==
        "UICrit_RICO"
    ):

        assert (
            row[
                "source_variable"
            ]
            in rico_names
        )

    if (
        row[
            "source_dataset"
        ]
        ==
        "Catching_UX_Flaws"
    ):

        assert (
            row[
                "source_variable"
            ]
            in catching_names
        )

print(
    "Mappings validados ✅"
)

Mappings validados ✅


In [13]:
full_records = []

for source in [
    "UICrit_RICO",
    "Catching_UX_Flaws"
]:

    source_mapping = (
        declared_mappings[
            declared_mappings[
                "source_dataset"
            ]
            ==
            source
        ]
    )

    for canonical in (
        canonical_x[
            "variable"
        ]
    ):

        matches = (
            source_mapping[
                source_mapping[
                    "canonical_variable"
                ]
                ==
                canonical
            ]
        )

        if matches.empty:

            full_records.append({
                "source_dataset":
                    source,

                "canonical_variable":
                    canonical,

                "source_variable":
                    None,

                "compatibility_level":
                    "NO_MAPPING",

                "possible_heuristic":
                    None,

                "reason":
                    "No se encontró una operacionalización compatible en esta fuente."
            })

        else:

            for _, mapping in (
                matches.iterrows()
            ):

                full_records.append(
                    mapping.to_dict()
                )

feature_compatibility_matrix = pd.DataFrame(
    full_records
)

In [14]:
feature_compatibility_matrix = (
    feature_compatibility_matrix
    .merge(
        compatibility_levels[
            [
                "compatibility_level",
                "can_be_renamed_canonical",
                "can_share_feature_space"
            ]
        ],
        on="compatibility_level",
        how="left",
        validate="many_to_one"
    )
)

In [15]:
display(
    feature_compatibility_matrix
)

,source_dataset,canonical_variable,source_variable,compatibility_level,possible_heuristic,reason,can_be_renamed_canonical,can_share_feature_space
0,UICrit_RICO,task_time_s,NaN,NO_MAPPING,NaN,No se encontró una operacionalización compatible en esta fuente.,False,False
1,UICrit_RICO,mean_response_time_ms,NaN,NO_MAPPING,NaN,No se encontró una operacionalización compatible en esta fuente.,False,False
2,UICrit_RICO,max_response_time_ms,NaN,NO_MAPPING,NaN,No se encontró una operacionalización compatible en esta fuente.,False,False
3,UICrit_RICO,mean_pre_action_pause_s,NaN,NO_MAPPING,NaN,No se encontró una operacionalización compatible en esta fuente.,False,False
4,UICrit_RICO,click_count,NaN,NO_MAPPING,NaN,No se encontró una operacionalización compatible en esta fuente.,False,False
5,UICrit_RICO,keypress_count,NaN,NO_MAPPING,NaN,No se encontró una operacionalización compatible en esta fuente.,False,False
6,UICrit_RICO,scroll_count,NaN,NO_MAPPING,NaN,No se encontró una operacionalización compatible en esta fuente.,False,False
7,UICrit_RICO,backtracking_count,NaN,NO_MAPPING,NaN,No se encontró una operacionalización compatible en esta fuente.,False,False
8,UICrit_RICO,repeated_action_count,NaN,NO_MAPPING,NaN,No se encontró una operacionalización compatible en esta fuente.,False,False
9,UICrit_RICO,navigation_steps,NaN,NO_MAPPING,NaN,No se encontró una operacionalización compatible en esta fuente.,False,False


In [16]:
compatibility_summary = (
    feature_compatibility_matrix
    .groupby(
        [
            "source_dataset",
            "compatibility_level"
        ]
    )
    .size()
    .reset_index(
        name="count"
    )
)

display(
    compatibility_summary
)

,source_dataset,compatibility_level,count
0,Catching_UX_Flaws,CONSTRUCT_PROXY,5
1,Catching_UX_Flaws,NO_MAPPING,34
2,Catching_UX_Flaws,PARTIAL,1
3,UICrit_RICO,CONSTRUCT_PROXY,5
4,UICrit_RICO,NO_MAPPING,35


In [17]:
exact_features = (
    feature_compatibility_matrix[
        feature_compatibility_matrix[
            "compatibility_level"
        ]
        ==
        "EXACT"
    ]
)

In [18]:
exact_per_source = (
    exact_features
    .groupby(
        "source_dataset"
    )[
        "canonical_variable"
    ]
    .apply(set)
    .to_dict()
)

rico_exact = exact_per_source.get(
    "UICrit_RICO",
    set()
)

catching_exact = exact_per_source.get(
    "Catching_UX_Flaws",
    set()
)

strict_common_space = (
    rico_exact
    &
    catching_exact
)

print(
    "EXACT RICO:",
    len(
        rico_exact
    )
)

print(
    "EXACT Catching:",
    len(
        catching_exact
    )
)

print(
    "Espacio común EXACT:",
    len(
        strict_common_space
    )
)

print(
    sorted(
        strict_common_space
    )
)

EXACT RICO: 0
EXACT Catching: 0
Espacio común EXACT: 0
[]


In [19]:
model_readiness = pd.DataFrame([
    {
        "model_or_experiment":
            "PRIMARY_NIELSEN_MODEL",

        "ready":
            False,

        "n_instances":
            0,

        "reason":
            "No existe todavía una fuente suficiente con X canónica compatible y Y Nielsen directa."
    },

    {
        "model_or_experiment":
            "UICRIT_RICO_PROXY",

        "ready":
            True,

        "n_instances":
            len(
                uicrit_rico_proxy
            ),

        "reason":
            "Permite un experimento proxy interno usando únicamente X y Y propias de UICrit/RICO."
    },

    {
        "model_or_experiment":
            "CATCHING_PIPELINE_VALIDATION",

        "ready":
            True,

        "n_instances":
            len(
                catching_validation
            ),

        "reason":
            "Permite verificar la extracción y trazabilidad X-Y, pero no evaluar clasificación."
    },

    {
        "model_or_experiment":
            "RICO_PLUS_CATCHING_ROW_UNION",

        "ready":
            False,

        "n_instances":
            0,

        "reason":
            "Las modalidades y operacionalizaciones de X no son equivalentes; no deben concatenarse como un único dataset."
    }
])

model_readiness

,model_or_experiment,ready,n_instances,reason
0,PRIMARY_NIELSEN_MODEL,False,0,No existe todavía una fuente suficiente con X canónica compatible y Y Nielsen directa.
1,UICRIT_RICO_PROXY,True,1000,Permite un experimento proxy interno usando únicamente X y Y propias de UICrit/RICO.
2,CATCHING_PIPELINE_VALIDATION,True,2,"Permite verificar la extracción y trazabilidad X-Y, pero no evaluar clasificación."
3,RICO_PLUS_CATCHING_ROW_UNION,False,0,Las modalidades y operacionalizaciones de X no son equivalentes; no deben concatenarse como un único dataset.


In [20]:
next_source_priority = pd.DataFrame([
    {
        "priority": 1,

        "dataset":
            "29042675",

        "objective":
            "Recuperar prototipos y determinar si existe X web correspondiente a Y Nielsen directa.",

        "reason":
            "Es actualmente la fuente abierta con mayor potencial para ampliar observaciones Nielsen directas y además contiene prototipos educativos."
    },

    {
        "priority": 2,

        "dataset":
            "AMUSED",

        "objective":
            "Reevaluar cuando se conceda acceso.",

        "reason":
            "Podría aportar X de interacción junto con etiquetas relacionadas."
    }
])

next_source_priority

,priority,dataset,objective,reason
0,1,29042675,Recuperar prototipos y determinar si existe X web correspondiente a Y Nielsen directa.,Es actualmente la fuente abierta con mayor potencial para ampliar observaciones Nielsen directas y además contiene prototipos educativos.
1,2,AMUSED,Reevaluar cuando se conceda acceso.,Podría aportar X de interacción junto con etiquetas relacionadas.


In [21]:
declared_mappings.to_csv(
    REPORTS_DIR
    / "declared_feature_mappings.csv",
    index=False
)

feature_compatibility_matrix.to_csv(
    REPORTS_DIR
    / "feature_compatibility_matrix.csv",
    index=False
)

compatibility_summary.to_csv(
    REPORTS_DIR
    / "feature_compatibility_summary.csv",
    index=False
)

model_readiness.to_csv(
    REPORTS_DIR
    / "model_readiness.csv",
    index=False
)

next_source_priority.to_csv(
    REPORTS_DIR
    / "next_direct_source_priority.csv",
    index=False
)

In [22]:
for filename in [
    "declared_feature_mappings.csv",
    "feature_compatibility_matrix.csv",
    "feature_compatibility_summary.csv",
    "model_readiness.csv",
    "next_direct_source_priority.csv"
]:

    path = (
        REPORTS_DIR
        / filename
    )

    print(
        filename,
        "✅"
        if path.exists()
        else "❌"
    )

declared_feature_mappings.csv ✅
feature_compatibility_matrix.csv ✅
feature_compatibility_summary.csv ✅
model_readiness.csv ✅
next_direct_source_priority.csv ✅
